#   Pipeline de Nettoyage des Données GDELT — Bénin
## Hackathon Isheero x DataCamp 2026 — Team 8

Ce notebook implémente le pipeline de nettoyage des données GDELT extraites
depuis Google BigQuery pour le Bénin (code pays : `BN`).

### Objectifs
- Supprimer les doublons
- Gérer les valeurs nulles
- Normaliser `GoldsteinScale` et `AvgTone`
- Convertir `SQLDATE` en datetime
- Sauvegarder les données propres dans `/data/clean/`

### Input
`data/raw/bq-results-last-12-months.csv` — export BigQuery brut (61 colonnes)

### Output
- `data/clean/gdelt_benin_clean.csv`
- `data/clean/gdelt_benin_clean.parquet`

## 1. Imports & Configuration
Chargement des librairies nécessaires et définition des chemins d'accès aux données.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_FILE  = Path("../data/raw/bq-results-last-12-months.csv")
CLEAN_DIR = Path("../data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print(f"  Source  : {RAW_FILE}")
print(f"  Sortie  : {CLEAN_DIR}")


  Source  : ..\data\raw\bq-results-last-12-months.csv
  Sortie  : ..\data\clean


## 2. Chargement des données brutes

Lecture du fichier CSV exporté depuis BigQuery.
`low_memory=False` évite les warnings de type mixte sur les grandes colonnes.

In [2]:
df = pd.read_csv(RAW_FILE, low_memory=False)
print(f"Lignes : {len(df):,}  |  Colonnes : {len(df.columns)}")
df.head(3)

Lignes : 22,778  |  Colonnes : 61


,GLOBALEVENTID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1KnownGroupCode,Actor1EthnicCode,...,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_ADM1Code,ActionGeo_ADM2Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL
0,1248558842,20250609,202506,2025,2025.4356,BEN,COTONOU,BEN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.5,2.25,BN,20250609020000,https://english.news.cn/20250609/833e6be5b34f4...
1,1248561281,20250609,202506,2025,2025.4356,NGA,BENIN CITY,NGA,NaN,NaN,...,1,Benin,BN,BN,NaN,9.5,2.25,BN,20250609023000,https://punchng.com/ndlea-intercepts-drugs-dis...
2,1248560915,20250609,202506,2025,2025.4356,BEN,BENIN,BEN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.5,2.25,BN,20250609023000,https://punchng.com/ndlea-intercepts-drugs-dis...


Suppression doublons

## 3. Suppression des doublons
`GLOBALEVENTID` est l'identifiant unique de chaque événement dans GDELT.
On déduplique sur cette colonne pour éliminer les lignes identiques
issues d'exports multiples ou de redondances BigQuery.

In [3]:
before = len(df)
df = df.drop_duplicates(subset=["GLOBALEVENTID"])
print(f"Doublons supprimés : {before - len(df):,}  →  {len(df):,} lignes restantes")

Doublons supprimés : 0  →  22,778 lignes restantes


SQLDATE → datetime

## 4. Conversion SQLDATE → datetime
Dans GDELT, `SQLDATE` est encodé en entier format `YYYYMMDD` (ex: `20250427`).
On le convertit en `datetime64` pour permettre les analyses temporelles.

On extrait aussi `year`, `month`, `week` comme colonnes dérivées
utiles pour les agrégations et visualisations.

In [4]:
df["SQLDATE"] = pd.to_datetime(df["SQLDATE"].astype(str), format="%Y%m%d", errors="coerce")
df["year"]    = df["SQLDATE"].dt.year
df["month"]   = df["SQLDATE"].dt.month
df["week"]    = df["SQLDATE"].dt.isocalendar().week.astype("Int64")

print(f"Période : {df['SQLDATE'].min().date()} → {df['SQLDATE'].max().date()}")
df[["SQLDATE", "year", "month", "week"]].head(3)

Période : 2025-04-26 → 2026-04-26


,SQLDATE,year,month,week
0,2025-06-09,2025,6,24
1,2025-06-09,2025,6,24
2,2025-06-09,2025,6,24


 Normalisation GoldsteinScale & AvgTone

## 5. Normalisation — GoldsteinScale & AvgTone

| Colonne | Description | Plage théorique |
|---|---|---|
| `GoldsteinScale` | Score de stabilité de l'événement | [-10, +10] |
| `AvgTone` | Ton moyen des articles sources | pas de borne stricte |

**Traitement appliqué :**
- Forcer le type `float` (coerce les erreurs → NaN)
- `GoldsteinScale` : clip entre -10 et +10
- Imputation des NaN par la **médiane** (plus robuste que la moyenne sur données asymétriques)

In [5]:
for col, low, high in [("GoldsteinScale", -10.0, 10.0), ("AvgTone", None, None)]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    if low  is not None: df[col] = df[col].clip(lower=low)
    if high is not None: df[col] = df[col].clip(upper=high)
    median     = df[col].median()
    nans       = df[col].isna().sum()
    df[col]    = df[col].fillna(median)
    print(f"{col:20s}  NaN imputés={nans}  médiane={median:.4f}")

df[["GoldsteinScale", "AvgTone"]].describe()

GoldsteinScale        NaN imputés=0  médiane=1.9000
AvgTone               NaN imputés=0  médiane=-1.6194


,GoldsteinScale,AvgTone
count,22778.000000,22778.000000
mean,0.500593,-1.611982
std,4.555559,4.532620
min,-10.000000,-18.518519
25%,-2.000000,-4.812834
50%,1.900000,-1.619433
75%,3.400000,1.750000
max,10.000000,19.480519


Gestion colonnes nulles 

## 6. Gestion des valeurs nulles — colonnes texte
Les colonnes acteurs et géographiques contiennent de nombreux NaN
car tous les événements n'ont pas deux acteurs identifiés.
On remplace ces NaN par `"Unknown"` pour éviter les erreurs
dans les analyses et visualisations ultérieures.

In [6]:
TEXT_COLS = [
    "Actor1Code", "Actor1Name", "Actor1CountryCode",
    "Actor1Type1Code", "Actor1Type2Code", "Actor1Type3Code",
    "Actor2Code", "Actor2Name", "Actor2CountryCode",
    "Actor2Type1Code", "Actor2Type2Code", "Actor2Type3Code",
    "ActionGeo_FullName", "Actor1Geo_FullName", "Actor2Geo_FullName",
]
for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")


print(f"✓ Colonnes texte nettoyées : {len(TEXT_COLS)}")
print(f"  NaN restants (total dataset) : {df.isna().sum().sum():,}")

✓ Colonnes texte nettoyées : 15
  NaN restants (total dataset) : 286,797


## 7. Rapport qualité final
Vérification globale avant sauvegarde.

In [7]:
print("═" * 50)
print("  RAPPORT QUALITÉ — Données nettoyées")
print("═" * 50)
print(f"  Lignes        : {len(df):,}")
print(f"  Colonnes      : {len(df.columns)}")
print(f"  Période       : {df['SQLDATE'].min().date()} → {df['SQLDATE'].max().date()}")
print(f"  NaN restants  : {df.isna().sum().sum():,}")
print()
for col in ["GoldsteinScale", "AvgTone"]:
    print(f"  {col:20s}  min={df[col].min():.2f}  max={df[col].max():.2f}  moy={df[col].mean():.2f}")
print("═" * 50)

══════════════════════════════════════════════════
  RAPPORT QUALITÉ — Données nettoyées
══════════════════════════════════════════════════
  Lignes        : 22,778
  Colonnes      : 64
  Période       : 2025-04-26 → 2026-04-26
  NaN restants  : 286,797

  GoldsteinScale        min=-10.00  max=10.00  moy=0.50
  AvgTone               min=-18.52  max=19.48  moy=-1.61
══════════════════════════════════════════════════


sauvegarde ds data/clean

## 8. Sauvegarde dans `/data/clean/`
Export en deux formats :
- **CSV** : lisible universellement, pour partage et vérification rapide
- **Parquet** : format colonnaire compressé, lecture rapide dans les notebooks d'analyse

In [11]:
csv_path     = CLEAN_DIR / "gdelt_benin_clean.csv"
parquet_path = CLEAN_DIR / "gdelt_benin_clean.parquet"

df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path, index=False, engine="pyarrow")

print(f"✓ CSV     sauvegardé : {csv_path}")
print(f"✓ Parquet sauvegardé : {parquet_path}")
print(f"  Taille CSV     : {csv_path.stat().st_size / 1e6:.1f} MB")
print(f"  Taille Parquet : {parquet_path.stat().st_size / 1e6:.1f} MB")

✓ CSV     sauvegardé : ..\data\clean\gdelt_benin_clean.csv
✓ Parquet sauvegardé : ..\data\clean\gdelt_benin_clean.parquet
  Taille CSV     : 9.0 MB
  Taille Parquet : 1.4 MB
